# Image and System Analysis | Division of Medical Radiation Physics | Stockholm University
```mehdi.astaraki@fysik.su.se```

# Image Compression, Information Theory, and Transform Coding
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Astarakee/isa-su/blob/main/labs/06_ImageCompression.ipynb)

**Course**: Image and System Analysis

**Level**: Undergraduate / Graduate Computational Lab

**Target Audience**: Medical Physicists, Computational Researchers, Biomedical Engineers, Image and Signal Processing Students

**Author**: `Mehdi Astaraki`

---

## Overview & Learning Objectives
Image compression is the art and science of reducing the number of bits required to represent digital imagery without unacceptable degradation of visual quality. Digital images exhibit three principal forms of data redundancy: **coding redundancy** (suboptimal allocation of variable-length codes to discrete intensity distributions), **inter-pixel (spatial) redundancy** (strong statistical autocorrelation between spatially adjacent pixels), and **psycho-visual redundancy** (visual details that the Human Visual System (HVS) cannot perceive).

By completing this notebook, you will master:
1. **Information Theory**: Shannon entropy ($H_0$), conditional entropy $H(X_2|X_1)$, source coding theorems, and Huffman coding with Kraft-McMillan verification.
2. **Spatial Decorrelation**: Differential Pulse Code Modulation (DPCM), Gray-coded bit-plane decomposition, and Run-Length Encoding (RLE).
3. **Distortion Metrology**: Objective mathematical metrics (MSE, RMSE, PSNR) vs. perceptual structural metrics (SSIM and local SSIM maps).
4. **Block-Based Transform Coding (JPEG)**: 2D Type-II Discrete Cosine Transform (DCT) basis decomposition, energy compaction, ISO/IEC $Q_{50}$ psycho-visual quantization matrix scaling, and zig-zag serialization.
5. **Rate-Distortion Analysis & Compression Artifacts**: Full-image block JPEG codec, empirical R-D curves, and low-bitrate blocking, ringing, and blurring artifacts.
6. **Multi-Resolution Wavelet Coding (JPEG 2000)**: 1-level 2D Haar DWT from scratch, dyadic Mallat multi-scale pyramids via Cohen-Daubechies-Feauveau 9/7 (`bior4.4`) wavelets, and hard/soft coefficient shrinkage.
7. **Advanced Modern Codec Technologies (23.1–23.11)**:
   - Arithmetic Coding
   - First-Order Context Modeling on Prediction Residuals
   - Lempel-Ziv-Welch (LZW) Dictionary Compression
   - PNG Adaptive Row Scanline Filtering (Sub, Up, Average, Paeth)
   - Lossless JPEG-LS Median Edge Detection (MED)
   - Quantizer Design (Uniform, Deadzone & Lloyd-Max)
   - Reverse Water-Filling Subband Bit Allocation
   - Rate-Distortion Optimization (RDO) Lagrangian Decision Modeling
   - Embedded Zerotree Wavelets (EZW / SPIHT) Progressive Coding
   - Modern Codec Bake-Off & Bjøntegaard-Delta Rate (BD-Rate)
   - Perceptual Quality Beyond SSIM (MS-SSIM, VIF)

---


---
## SECTION 0: Environment Setup & Library Loader

### Theoretical & Engineering Setup
Transform coding and statistical rate-distortion calculations require floating-point arithmetic (`np.float64`) and standardized benchmark image matrices. In this section, we import runtime libraries (`numpy`, `scipy`, `matplotlib`, `skimage`, `cv2`, `pywt`), set publication-quality inline plotting defaults, and build a dictionary of standard test targets from `skimage.data`.


In [ ]:
# Section 0: Environment Setup & Library Loader
import os
import sys
import heapq
import numpy as np
import scipy.ndimage as ndimage
import scipy.signal as signal
from scipy import fftpack, io
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D
from skimage import data, color, transform, filters, exposure, util, metrics
import cv2
from PIL import Image

# Automated package installation for Colab
try:
    import pywt
except ImportError:
    print("Installing PyWavelets...")
    os.system(f"{sys.executable} -m pip install -q PyWavelets")
    import pywt

# Configure publication-grade styling
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 9.5
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 10.5
plt.rcParams['xtick.labelsize'] = 8.5
plt.rcParams['ytick.labelsize'] = 8.5
plt.rcParams['image.cmap'] = 'gray'

def load_compression_datasets():
    datasets = {
        'camera': data.camera(),
        'coins': data.coins(),
        'checkerboard': data.checkerboard(),
        'text': data.text(),
        'astronaut': color.rgb2gray(data.astronaut()),
        'chelsea': color.rgb2gray(data.chelsea()),
        'page': data.page()
    }
    return datasets

img_assets = load_compression_datasets()
print(f"Environment successfully initialized. Available datasets: {list(img_assets.keys())}")


---
## SECTION 1: Compression Metrics, Data Rates, and Redundancy

### 1.1 Mathematical Formulation of Data Rates
Let an uncompressed monochromatic digital image $f(x,y)$ of size $M \times N$ be represented using $b$ total bits (for an 8-bit image, $b = 8 \times M \times N$ bits). Let the compressed representation occupy $b'$ bits on disk.

We formulate the fundamental compression metrics:
1. **Compression Ratio ($C_R$)**:
   $$C_R = \frac{b}{b'}$$
2. **Relative Data Redundancy ($R_D$)**:
   $$R_D = 1 - \frac{1}{C_R} = \frac{b - b'}{b}$$
   - If $C_R = 1 \implies R_D = 0$ (No redundancy removed).
   - If $C_R = 10 \implies R_D = 0.90$ ($90\%$ of the data bits were redundant).
   - If $b' \to 0 \implies C_R \to \infty, R_D \to 1.0$.
3. **Average Bitrate ($\text{bpp}$ - Bits Per Pixel)**:
   $$\text{bpp} = \frac{b'}{M \times N} = \frac{8}{C_R} \quad [\text{bits/pixel}]$$


In [ ]:
# Cell 1.1 & 1.2: Measurement of Real File Byte Allocations and Compression Metrics
img_cam = img_assets['camera']
M, N = img_cam.shape
uncompressed_bytes = M * N
uncompressed_bits = uncompressed_bytes * 8

# Encode to in-memory byte buffers using lossless PNG and lossy JPEG
_, buf_png = cv2.imencode('.png', img_cam)
_, buf_jpeg_q80 = cv2.imencode('.jpg', img_cam, [int(cv2.IMWRITE_JPEG_QUALITY), 80])
_, buf_jpeg_q15 = cv2.imencode('.jpg', img_cam, [int(cv2.IMWRITE_JPEG_QUALITY), 15])

bytes_png = len(buf_png)
bytes_jpeg_q80 = len(buf_jpeg_q80)
bytes_jpeg_q15 = len(buf_jpeg_q15)

def calc_metrics(b_raw, b_comp, M, N):
    cr = b_raw / b_comp
    rd = 1.0 - (1.0 / cr)
    bpp = (b_comp * 8) / (M * N)
    return cr, rd, bpp

cr_png, rd_png, bpp_png = calc_metrics(uncompressed_bytes, bytes_png, M, N)
cr_j80, rd_j80, bpp_j80 = calc_metrics(uncompressed_bytes, bytes_jpeg_q80, M, N)
cr_j15, rd_j15, bpp_j15 = calc_metrics(uncompressed_bytes, bytes_jpeg_q15, M, N)

print(f"=== Compression Metrics on Cameraman ({M}x{N}) ===")
print(f"Raw Uncompressed:  {uncompressed_bytes:,} bytes | 8.00 bpp | CR = 1.00:1 | RD = 0.00%")
print(f"Lossless PNG:      {bytes_png:,} bytes | {bpp_png:.2f} bpp | CR = {cr_png:.2f}:1 | RD = {rd_png*100:.1f}%")
print(f"Lossy JPEG (Q=80): {bytes_jpeg_q80:,} bytes | {bpp_j80:.2f} bpp | CR = {cr_j80:.2f}:1 | RD = {rd_j80*100:.1f}%")
print(f"Lossy JPEG (Q=15): {bytes_jpeg_q15:,} bytes | {bpp_j15:.2f} bpp | CR = {cr_j15:.2f}:1 | RD = {rd_j15*100:.1f}%")


---
## SECTION 2: Statistical Information Theory & Coding Redundancy

### 2.1 Shannon Zeroth-Order Information Entropy
Claude Shannon (1948) proved that the theoretical minimum average bitrate required to represent a memoryless discrete information source without loss is its **Information Entropy ($H_0$)**:
$$H_0 = -\sum_{k=0}^{L-1} p_k \log_2(p_k) \quad [\text{bits/pixel}]$$
where:
- $L = 256$ is the total alphabet size for 8-bit grayscale imagery.
- $p_k = \frac{n_k}{M \times N}$ is the normalized empirical probability of intensity symbol $k$.
- By definition, $\lim_{p_k \to 0} p_k \log_2(p_k) = 0$.

### 2.2 Coding Redundancy Gap
If each pixel is stored using a fixed-length $b = 8\text{ bits}$ allocation, the **Coding Redundancy ($R_C$)** is:
$$R_C = b - H_0 = 8 - H_0 \quad [\text{bits/pixel}]$$
$R_C$ represents the exact number of bits wasted per pixel due to using uniform fixed-length words rather than assigning shorter codes to more frequent symbols.


In [ ]:
# Cell 2.1 & 2.2: Computation of Shannon Entropy and Coding Redundancy
def compute_entropy_and_redundancy(img):
    hist, _ = np.histogram(img.ravel(), bins=256, range=(0, 256))
    pk = hist / np.sum(hist)
    pk_nonzero = pk[pk > 0]
    H0 = -np.sum(pk_nonzero * np.log2(pk_nonzero))
    Rc = 8.0 - H0
    return H0, Rc, pk

H0_cam, Rc_cam, pk_cam = compute_entropy_and_redundancy(img_assets['camera'])
H0_text, Rc_text, pk_text = compute_entropy_and_redundancy(img_assets['text'])

fig, axes = plt.subplots(2, 2, figsize=(15, 8.5))

axes[0, 0].imshow(img_assets['camera'], cmap='gray')
axes[0, 0].set_title("(A) Continuous-Tone Photographic Scene (Cameraman)", fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].bar(range(256), pk_cam, color='steelblue', width=1.0)
axes[0, 1].set_title(f"(B) Intensity Histogram: $H_0 = {H0_cam:.3f}$ bpp | $R_C = {Rc_cam:.3f}$ bpp", fontweight='bold')
axes[0, 1].set_xlabel("Intensity Level $k \in [0, 255]$")
axes[0, 1].set_ylabel("Probability $p_k$")
axes[0, 1].grid(True, linestyle='--', alpha=0.6)

axes[1, 0].imshow(img_assets['text'], cmap='gray')
axes[1, 0].set_title("(C) High-Contrast Document Scene (Text)", fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].bar(range(256), pk_text, color='darkorange', width=1.0)
axes[1, 1].set_title(f"(D) Intensity Histogram: $H_0 = {H0_text:.3f}$ bpp | $R_C = {Rc_text:.3f}$ bpp", fontweight='bold')
axes[1, 1].set_xlabel("Intensity Level $k \in [0, 255]$")
axes[1, 1].set_ylabel("Probability $p_k$")
axes[1, 1].grid(True, linestyle='--', alpha=0.6)

plt.suptitle("Figure 2.1: Shannon Zeroth-Order Information Entropy and Coding Redundancy Discrepancies",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 3: Inter-Pixel Spatial Redundancy & Correlation Decay

### 3.1 Inter-Pixel Pearson Autocorrelation
Natural scenes exhibit strong geometric continuity: adjacent pixels $f(x,y)$ and $f(x+1, y)$ carry nearly identical radiometric energy. The horizontal Pearson correlation coefficient is:
$$\rho = \frac{\sum_{x=0}^{M-2}\sum_{y=0}^{N-1} (f(x,y) - \mu)(f(x+1,y) - \mu)}{\sigma^2}$$

### 3.2 1D Autocorrelation Decay Profile
The spatial autocorrelation function over lag distance $\Delta x$ is modeled as a first-order Markov process:
$$R_{ff}(\Delta x) = \mathbb{E}\big[ (f(x,y) - \mu)(f(x+\Delta x, y) - \mu) \big] \approx \sigma^2 \rho^{|\Delta x|}$$
In natural photography, $\rho \ge 0.95$, confirming that individual pixel intensities are highly predictable from their local spatial neighborhood.


In [ ]:
# Cell 3.1 & 3.2: 2D Spatial Scatter Plot and Autocorrelation Decay Profiles
img_cam_f = img_assets['camera'].astype(np.float64)
p_curr = img_cam_f[:, :-1].ravel()
p_next = img_cam_f[:, 1:].ravel()

# Compute Pearson Correlation
rho_val = np.corrcoef(p_curr, p_next)[0, 1]

# Compute Autocorrelation over lag distances Delta x in [0, 50]
lags = np.arange(0, 51)
autocorr = np.zeros(len(lags))
mu_val = np.mean(img_cam_f)
var_val = np.var(img_cam_f)

for idx, delta_x in enumerate(lags):
    if delta_x == 0:
        autocorr[idx] = 1.0
    else:
        c1 = img_cam_f[:, :-delta_x] - mu_val
        c2 = img_cam_f[:, delta_x:] - mu_val
        autocorr[idx] = np.mean(c1 * c2) / var_val

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Subsample scatter points for clean plotting
sub_idx = np.random.choice(len(p_curr), size=8000, replace=False)
axes[0].scatter(p_curr[sub_idx], p_next[sub_idx], c='steelblue', s=8, alpha=0.4, edgecolors='none')
axes[0].plot([0, 255], [0, 255], 'r--', linewidth=2, label='Identity Line ($f_{x+1} = f_x$)')
axes[0].set_title(f"(A) Adjacent Horizontal Pixel Scatter\n[Pearson Correlation $\\rho = {rho_val:.4f}$]", fontweight='bold')
axes[0].set_xlabel("Pixel Intensity $f(x, y)$")
axes[0].set_ylabel("Adjacent Pixel Intensity $f(x+1, y)$")
axes[0].set_xlim(0, 255)
axes[0].set_ylim(0, 255)
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend(loc='upper left')

axes[1].plot(lags, autocorr, 'b.-', linewidth=2.0, markersize=6, label='Empirical $R_{ff}(\Delta x)$')
axes[1].plot(lags, rho_val**lags, 'r--', linewidth=1.8, label=r'Markov Model $\rho^{|\Delta x|}$')
axes[1].set_title("(B) Radial Spatial Autocorrelation Decay Profile", fontweight='bold')
axes[1].set_xlabel("Spatial Lag Distance $\Delta x$ (Pixels)")
axes[1].set_ylabel("Normalized Autocorrelation")
axes[1].set_xlim(0, 50)
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(loc='upper right')

plt.suptitle("Figure 3.1: Inter-Pixel Spatial Redundancy: High Adjacent Correlation and Spatial Autocorrelation Decay",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 4: The Deterministic Checkerboard Paradox & Conditional Entropy

### 4.1 The Checkerboard Paradox
Consider a binary $256 \times 256$ checkerboard where pixels alternate deterministically between $0$ (Black) and $1$ (White).
- Since exactly $50\%$ of pixels are $0$ and $50\%$ are $1$ ($p_0 = p_1 = 0.5$):
  $$H_0 = -0.5 \log_2(0.5) - 0.5 \log_2(0.5) = 1.000 \quad [\text{bits/pixel}]$$
- Naive zeroth-order entropy indicates that the image contains **maximum uncertainty** (1 bit per pixel). However, the image is **completely deterministic**: knowing pixel $f(x,y)$ reveals $f(x+1,y) = 1 - f(x,y)$ with $100\%$ certainty!

### 4.2 First-Order Conditional Entropy
To measure structural predictability, we evaluate **First-Order Conditional Entropy**:
$$H(X_2 \mid X_1) = -\sum_{i=0}^1 \sum_{j=0}^1 p(x_i, x_j) \log_2 p(x_j \mid x_i)$$
For the checkerboard, $p(0|0) = 0, p(1|0) = 1, p(0|1) = 1, p(1|1) = 0$.
$$H(X_2 \mid X_1) = 0.000 \quad [\text{bits/pixel}]$$
This proves that zeroth-order entropy fails on spatially correlated data, necessitating **predictive decorrelation**.


In [ ]:
# Cell 4.1 & 4.2: Evaluation of the Checkerboard Paradox
# Synthesize a 256x256 checkerboard alternating at every pixel: f(x,y) = (x + y) mod 2
yy, xx = np.indices((256, 256))
checker = ((xx + yy) % 2).astype(int)

# 1. Zeroth-Order Entropy
p_0 = np.mean(checker == 0)
p_1 = np.mean(checker == 1)
H0_chk = -(p_0 * np.log2(p_0) + p_1 * np.log2(p_1))

# 2. First-Order Transition Probabilities
x1 = checker[:, :-1].ravel()
x2 = checker[:, 1:].ravel()

N_total = len(x1)
p_joint = np.zeros((2, 2))
for i in [0, 1]:
    for j in [0, 1]:
        p_joint[i, j] = np.sum((x1 == i) & (x2 == j)) / N_total

# Conditional Entropy H(X2 | X1)
H_cond = 0.0
for i in [0, 1]:
    p_xi = np.sum(p_joint[i, :])
    if p_xi > 0:
        for j in [0, 1]:
            if p_joint[i, j] > 0:
                p_cond = p_joint[i, j] / p_xi
                H_cond -= p_joint[i, j] * np.log2(p_cond)

print(f"=== The Checkerboard Paradox ===")
print(f"Marginal Zeroth-Order Entropy H0:     {H0_chk:.4f} bpp (Maximal Uncertainty)")
print(f"First-Order Conditional Entropy H(X2|X1): {H_cond:.4f} bpp (Zero Uncertainty)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].imshow(checker[:32, :32], cmap='gray', interpolation='nearest')
axes[0].set_title("(A) Binary Checkerboard (Zoomed $32\\times 32$)\n[Alternating Deterministic Pattern]", fontweight='bold')
axes[0].axis('off')

im1 = axes[1].imshow(p_joint, cmap='Blues', vmin=0, vmax=0.6)
axes[1].set_title(r"(B) Joint State Transition Matrix $p(x_i, x_j)$" + "\n" + f"[$H_0 = {H0_chk:.2f}$ bpp vs. $H(X_2|X_1) = {H_cond:.2f}$ bpp]", fontweight='bold', color='darkgreen')
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(['$X_2=0$', '$X_2=1$'])
axes[1].set_yticklabels(['$X_1=0$', '$X_1=1$'])
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f"{p_joint[i, j]:.3f}", ha='center', va='center', color='crimson', fontsize=12, fontweight='bold')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

plt.suptitle("Figure 4.1: The Deterministic Checkerboard Paradox: Resolving Zeroth-Order Entropy Limitations via Context",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 5: Predictive Decorrelation (DPCM) & Residual Entropy

### 5.1 Differential Pulse Code Modulation (DPCM)
Differential Pulse Code Modulation decorrelates an image by subtracting a spatial prediction $\hat{f}(x,y)$ from the true pixel value $f(x,y)$:
$$\hat{f}(x,y) = f(x, y-1) \quad (\text{1D Left-Neighbor Predictor})$$
$$e(x,y) = f(x,y) - \hat{f}(x,y)$$

### 5.2 Laplacian Residual Compaction
While raw photographic intensities follow broad multimodal distributions, prediction errors $e(x,y)$ follow a zero-mean **Laplacian (double-exponential) distribution**:
$$p(e) = \frac{1}{2b} \exp\left( -\frac{|e|}{b} \right)$$
Because most prediction errors are near zero, the **Residual Entropy $H_{\text{residual}}$** drops drastically below the raw entropy $H_0$.


In [ ]:
# Cell 5.1 & 5.2: Vectorized 1D DPCM and Residual Entropy Reduction
def apply_1d_dpcm(img):
    img_f = img.astype(np.float64)
    # Predictor: Left neighbor
    pred = np.zeros_like(img_f)
    pred[:, 1:] = img_f[:, :-1]
    pred[:, 0] = 128.0 # Baseline for first column
    
    residual = img_f - pred
    return residual, pred

res_cam, _ = apply_1d_dpcm(img_assets['camera'])
res_coins, _ = apply_1d_dpcm(img_assets['coins'])

def compute_residual_entropy(res):
    # Integer quantized residuals
    res_int = np.round(res).astype(int)
    unique, counts = np.unique(res_int, return_counts=True)
    pk = counts / np.sum(counts)
    return -np.sum(pk * np.log2(pk))

H_raw_cam, _, _ = compute_entropy_and_redundancy(img_assets['camera'])
H_res_cam = compute_residual_entropy(res_cam)

H_raw_coins, _, _ = compute_entropy_and_redundancy(img_assets['coins'])
H_res_coins = compute_residual_entropy(res_coins)

print(f"=== Predictive Decorrelation (DPCM) Entropy Savings ===")
print(f"Cameraman: Raw H0 = {H_raw_cam:.3f} bpp -> Residual H = {H_res_cam:.3f} bpp (Saving: {H_raw_cam - H_res_cam:.3f} bpp)")
print(f"Coins:     Raw H0 = {H_raw_coins:.3f} bpp -> Residual H = {H_res_coins:.3f} bpp (Saving: {H_raw_coins - H_res_coins:.3f} bpp)")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].imshow(img_assets['camera'], cmap='gray')
axes[0].set_title(f"(A) Original Image ($H_0 = {H_raw_cam:.2f}$ bpp)", fontweight='bold')
axes[0].axis('off')

# Display centered residual
axes[1].imshow(res_cam, cmap='gray', vmin=-60, vmax=60)
axes[1].set_title(f"(B) DPCM Error $e(x,y)$ ($H_{{\\mathrm{{res}}}} = {H_res_cam:.2f}$ bpp)", fontweight='bold', color='darkgreen')
axes[1].axis('off')

# Plot Laplacian Histogram
axes[2].hist(res_cam.ravel(), bins=100, range=(-50, 50), color='forestgreen', edgecolor='black', density=True)
axes[2].set_title("(C) Residual Error Laplacian PDF $p(e)$", fontweight='bold')
axes[2].set_xlabel("Prediction Error $e = f - \hat{f}$")
axes[2].set_ylabel("Probability Density")
axes[2].grid(True, linestyle='--', alpha=0.6)

plt.suptitle("Figure 5.1: Predictive Decorrelation (DPCM): Energy Compaction into Zero-Mean Laplacian Residual",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 6: Huffman Coding from Scratch & Kraft-McMillan Verification

### 6.1 Huffman Algorithm Mechanics
David Huffman (1952) formulated the optimal prefix-free variable-length code construction:
1. **Min-Heap Queue**: Initialize a priority queue of leaf nodes weighted by symbol probabilities $p_k$.
2. **Bottom-Up Merge**: Iteratively extract the two nodes with the lowest probabilities, merge them into a parent node with weight $p_i + p_j$, and re-insert into the queue until a single root node remains.
3. **Prefix-Free Code Assignment**: Traverse from root to leaves, assigning `'0'` to left branches and `'1'` to right branches.

### 6.2 Shannon's Source Coding Theorem & Kraft-McMillan Inequality
- **Shannon Bound**: The average code length $L_{\text{avg}} = \sum_{k=0}^{L-1} p_k l_k$ is bounded by:
  $$H_0 \le L_{\text{avg}} < H_0 + 1 \quad [\text{bits/symbol}]$$
- **Kraft-McMillan Inequality**: A prefix-free code is uniquely decodable if and only if:
  $$K = \sum_{k=0}^{L-1} 2^{-l_k} \le 1.0$$


In [ ]:
# Cell 6.1 & 6.2: Huffman Codec Implementation and Kraft-McMillan Verification
class HuffmanNode:
    def __init__(self, symbol=None, freq=0, left=None, right=None):
        self.symbol = symbol
        self.freq = freq
        self.left = left
        self.right = right
    
    def __lt__(self, other):
        return self.freq < other.freq

class HuffmanCodec:
    def __init__(self):
        self.codebook = {}
        self.reverse_codebook = {}
        self.root = None
        
    def build_tree(self, probabilities):
        heap = []
        for symbol, prob in probabilities.items():
            if prob > 0:
                heapq.heappush(heap, HuffmanNode(symbol=symbol, freq=prob))
                
        while len(heap) > 1:
            node1 = heapq.heappop(heap)
            node2 = heapq.heappop(heap)
            merged = HuffmanNode(freq=node1.freq + node2.freq, left=node1, right=node2)
            heapq.heappush(heap, merged)
            
        self.root = heap[0]
        self._generate_codes(self.root, "")
        
    def _generate_codes(self, node, current_code):
        if node is None:
            return
        if node.symbol is not None:
            self.codebook[node.symbol] = current_code
            self.reverse_codebook[current_code] = node.symbol
            return
        self._generate_codes(node.left, current_code + "0")
        self._generate_codes(node.right, current_code + "1")
        
    def encode(self, sequence):
        return "".join([self.codebook[sym] for sym in sequence])

# Test on Cameraman Intensity Distribution
symbols, counts = np.unique(img_assets['camera'].ravel(), return_counts=True)
probs = {sym: count / np.sum(counts) for sym, count in zip(symbols, counts)}

codec = HuffmanCodec()
codec.build_tree(probs)

# Calculate Average Length L_avg and Kraft Sum
L_avg = sum(probs[sym] * len(codec.codebook[sym]) for sym in probs)
kraft_sum = sum(2.0**(-len(code)) for code in codec.codebook.values())

print(f"=== Huffman Coding Verification on Cameraman ===")
print(f"Zeroth-Order Shannon Entropy H0: {H0_cam:.4f} bits/symbol")
print(f"Huffman Average Code Length L_avg: {L_avg:.4f} bits/symbol")
print(f"Coding Efficiency (H0 / L_avg):   {(H0_cam / L_avg)*100:.2f}%")
print(f"Kraft-McMillan Inequality Sum K:  {kraft_sum:.6f} (Must be <= 1.0)")
assert kraft_sum <= 1.0 + 1e-12, "Kraft-McMillan inequality violated!"
print("Assertion Passed: Code is uniquely decodable and satisfies the Kraft inequality.")


---
## SECTION 7: Bit-Plane Decomposition & Run-Length Encoding (RLE)

### 7.1 Bit-Plane Slicing
An 8-bit digital image $f(x,y)$ decomposes into eight binary bit planes $b_0(x,y), \dots, b_7(x,y)$:
$$f(x,y) = \sum_{k=0}^7 b_k(x,y) 2^k = b_0 2^0 + b_1 2^1 + \dots + b_7 2^7$$
- **Most Significant Bits (MSBs: $b_7, b_6$)**: Contain $>75\%$ of the total perceptual structural energy and form large uniform spatial areas.
- **Least Significant Bits (LSBs: $b_0, b_1$)**: Exhibit high entropy and behave like random noise.

### 7.2 Run-Length Encoding (RLE)
Run-Length Encoding compresses binary strings by storing consecutive identical values as pairs $(v_i, l_i)$ of value and run length. It achieves high compression on MSBs ($b_7$) but expands low-order bit planes ($b_0$).


In [ ]:
# Cell 7.1 & 7.2: Bit-Plane Decomposition and RLE Compression Sweep
def decompose_bit_planes(img):
    planes = []
    for k in range(8):
        plane = (img >> k) & 1
        planes.append(plane)
    return planes

def run_length_encode_1d(binary_array):
    # Vectorized 1D RLE
    flat = binary_array.ravel()
    n = len(flat)
    if n == 0:
        return 0
    # Locate transition indices
    transitions = np.where(flat[:-1] != flat[1:])[0] + 1
    run_starts = np.r_[0, transitions]
    run_ends = np.r_[transitions, n]
    run_lengths = run_ends - run_starts
    # Each run stored as (bit_val, 8-bit count) -> 9 bits total per run
    compressed_bits = len(run_lengths) * 9
    return compressed_bits

planes = decompose_bit_planes(img_assets['camera'])

fig, axes = plt.subplots(2, 4, figsize=(16, 7.5))
rle_bitrates = []

for k in range(8):
    ax = axes[1 - (k // 4), k % 4]
    ax.imshow(planes[k], cmap='gray')
    comp_bits = run_length_encode_1d(planes[k])
    cr_k = (M * N) / comp_bits
    rle_bitrates.append(comp_bits / (M * N))
    ax.set_title(f"Bit Plane $b_{k}$ (Weight $2^{k}$)\n[RLE CR = {cr_k:.2f}:1]", fontweight='bold')
    ax.axis('off')

plt.suptitle("Figure 7.1: 8-Bit Binary Bit-Plane Decomposition showing Structural Coherence Transition from MSB to LSB",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 8: Gray-Code Transformation for Bit-Plane Decorrelation

### 8.1 The Binary Hamming Transition Artifact
In standard binary representation, a minuscule spatial change of $+1$ gray level across a smooth gradient (e.g., $127 = 01111111_2 \to 128 = 10000000_2$) toggles all 8 bits simultaneously. This artificial bit flipping disrupts spatial coherence across all lower bit planes.

### 8.2 Binary-to-Gray Code Conversion
Gray codes guarantee that adjacent intensity levels differ by **exactly one bit**:
$$g = b \oplus (b \gg 1)$$
where $\oplus$ is the bitwise XOR operator. Converting an image to Gray-coded bit planes suppresses false edge transitions and significantly improves Run-Length Encoding efficiency.


In [ ]:
# Cell 8.1 & 8.2: Binary vs. Gray-Coded Bit Plane RLE Comparison
def binary_to_gray(img):
    return img ^ (img >> 1)

gray_coded_img = binary_to_gray(img_assets['camera'])
gray_planes = decompose_bit_planes(gray_coded_img)

bin_rle_sizes = [run_length_encode_1d(p) for p in planes]
gray_rle_sizes = [run_length_encode_1d(p) for p in gray_planes]

fig, ax = plt.subplots(figsize=(10, 4.5))

x_indices = np.arange(8)
width = 0.35

ax.bar(x_indices - width/2, [M*N/s for s in bin_rle_sizes], width, label='Standard Binary Planes', color='steelblue')
ax.bar(x_indices + width/2, [M*N/s for s in gray_rle_sizes], width, label='Gray-Coded Planes', color='forestgreen')
ax.set_title("Figure 8.1: Run-Length Compression Ratio Enhancement via Gray-Code Bit-Plane Conversion", fontweight='bold')
ax.set_xlabel("Bit Plane Index $k \in [0, 7]$")
ax.set_ylabel("RLE Compression Ratio $C_R$")
ax.set_xticks(x_indices)
ax.set_xticklabels([f'$b_{k}$' for k in range(8)])
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()


---
## SECTION 9: Mathematical Distortion Metrics: MSE, RMSE, and PSNR

### 9.1 Mathematical Formulations
Let $f(x,y)$ be the ground-truth image and $\hat{f}(x,y)$ be the compressed/reconstructed image of size $M \times N$:
1. **Mean Squared Error (MSE)**:
   $$\text{MSE} = \frac{1}{MN} \sum_{x=0}^{M-1} \sum_{y=0}^{N-1} \left[ f(x,y) - \hat{f}(x,y) \right]^2$$
2. **Root Mean Squared Error (RMSE)**:
   $$\text{RMSE} = \sqrt{\text{MSE}}$$
3. **Peak Signal-to-Noise Ratio (PSNR)**:
   $$\text{PSNR} = 10 \log_{10}\left( \frac{\text{MAX}_I^2}{\text{MSE}} \right) \quad [\text{dB}]$$
   where $\text{MAX}_I = 255.0$ for 8-bit imagery.


In [ ]:
# Cell 9.1 & 9.2: NumPy Distortion Engine and Scikit-Image Validation
def compute_mse(f, f_hat):
    return np.mean((f.astype(np.float64) - f_hat.astype(np.float64))**2)

def compute_rmse(f, f_hat):
    return np.sqrt(compute_mse(f, f_hat))

def compute_psnr(f, f_hat, max_val=255.0):
    mse = compute_mse(f, f_hat)
    if mse == 0:
        return float('inf')
    return 10.0 * np.log10((max_val**2) / mse)

# Test on simulated noisy image
np.random.seed(42)
img_test = img_assets['camera'].astype(np.float64)
img_recon = np.clip(img_test + np.random.normal(0, 12.0, img_test.shape), 0, 255)

mse_custom = compute_mse(img_test, img_recon)
psnr_custom = compute_psnr(img_test, img_recon)

mse_skimage = metrics.mean_squared_error(img_test, img_recon)
psnr_skimage = metrics.peak_signal_noise_ratio(img_test, img_recon, data_range=255.0)

print(f"=== Mathematical Distortion Metrics Validation ===")
print(f"Custom MSE:  {mse_custom:.4f} | Scikit-Image MSE:  {mse_skimage:.4f}")
print(f"Custom PSNR: {psnr_custom:.4f} dB | Scikit-Image PSNR: {psnr_skimage:.4f} dB")
assert np.isclose(mse_custom, mse_skimage) and np.isclose(psnr_custom, psnr_skimage), "Metric mismatch!"
print("Assertion Passed: Custom MSE and PSNR match standard scikit-image implementations.")


---
## SECTION 10: Psycho-Visual Pitfalls of PSNR vs. Structural Similarity (SSIM)

### 10.1 The Failure of PSNR as a Perceptual Metric
PSNR computes purely pixel-wise arithmetic differences. It is completely blind to spatial structures, textures, and human visual perception:
- A subtle **1-pixel spatial shift** destroys pixel-by-pixel alignment, resulting in a disastrously low PSNR despite being visually indistinguishable from the original.
- Adding **high-frequency Gaussian noise** of matched power generates essentially the same PSNR while severely degrading perceptual quality.

### 10.2 Structural Similarity Index (SSIM)
Wang, Bovik, Sheikh, and Simoncelli (2004) formulated **SSIM**, which measures structural preservation by decomposing local similarity into Luminance ($l$), Contrast ($c$), and Structure ($s$):
$$\text{SSIM}(x,y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}$$
where constants $C_1 = (0.01 \cdot 255)^2$ and $C_2 = (0.03 \cdot 255)^2$ prevent numerical instability.


In [ ]:
# Cell 10.1 & 10.2: Psycho-Visual Demonstration: Noise vs. Spatial Shift
img_orig = img_assets['camera'].astype(np.float64)

# Distortion 1: 1-Pixel translation
img_shifted = np.roll(np.roll(img_orig, 1, axis=0), 1, axis=1)

# Distortion 2: High-frequency Gaussian noise with power matched to the shift error (equal MSE)
np.random.seed(42)
noise_sigma = compute_rmse(img_orig, img_shifted)
img_noisy = np.clip(img_orig + np.random.normal(0, noise_sigma, img_orig.shape), 0, 255)

psnr_noisy = compute_psnr(img_orig, img_noisy)
psnr_shifted = compute_psnr(img_orig, img_shifted)

ssim_noisy = metrics.structural_similarity(img_orig, img_noisy, data_range=255.0)
ssim_shifted = metrics.structural_similarity(img_orig, img_shifted, data_range=255.0)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(img_orig, cmap='gray')
axes[0].set_title("(A) Reference Image", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img_noisy, cmap='gray')
axes[1].set_title(f"(B) Additive AWGN ($\sigma={noise_sigma:.1f}$)\n[PSNR: {psnr_noisy:.2f} dB | SSIM: {ssim_noisy:.4f}]",
                  fontweight='bold', color='crimson')
axes[1].axis('off')

axes[2].imshow(img_shifted, cmap='gray')
axes[2].set_title(f"(C) 1-Pixel Translation\n[PSNR: {psnr_shifted:.2f} dB | SSIM: {ssim_shifted:.4f}]",
                  fontweight='bold', color='darkgreen')
axes[2].axis('off')

plt.suptitle("Figure 10.1: The PSNR Paradox: Matched PSNR with Vastly Different Perceptual Quality",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 11: Local SSIM Structural Mapping

### 11.1 Local Windowed SSIM Maps
Rather than evaluating a single global scalar value, SSIM computes quality locally over an $11 \times 11$ circular Gaussian window ($\sigma = 1.5$) to produce a 2D spatial degradation map $\text{SSIM}(x,y) \in [-1.0, 1.0]$:
$$\text{SSIM}_{\text{map}}(x,y) = \frac{\big(2\mu_x(x,y)\mu_y(x,y) + C_1\big)\big(2\sigma_{xy}(x,y) + C_2\big)}{\big(\mu_x^2(x,y) + \mu_y^2(x,y) + C_1\big)\big(\sigma_x^2(x,y) + \sigma_y^2(x,y) + C_2\big)}$$


In [ ]:
# Cell 11.1 & 11.2: Local SSIM Quality Map Computation and Visualization
_, ssim_map_noisy = metrics.structural_similarity(img_orig, img_noisy, data_range=255.0, full=True)
sq_error_map = (img_orig - img_noisy)**2

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

axes[0].imshow(img_orig, cmap='gray')
axes[0].set_title("(A) Reference Image", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img_noisy, cmap='gray')
axes[1].set_title("(B) AWGN Degraded", fontweight='bold')
axes[1].axis('off')

im2 = axes[2].imshow(sq_error_map, cmap='hot')
axes[2].set_title("(C) Squared Error $(f - \hat{f})^2$", fontweight='bold')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

im3 = axes[3].imshow(ssim_map_noisy, cmap='viridis', vmin=0, vmax=1)
axes[3].set_title(r"(D) Local $\text{SSIM}(x,y)$ Map", fontweight='bold', color='darkgreen')
axes[3].axis('off')
plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

plt.suptitle("Figure 11.1: Spatial Error Maps: Unweighted Squared Euclidean Error vs. Perceptual Local SSIM Map",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 12: Visualizing the 2D Discrete Cosine Transform (DCT) Basis

### 12.1 The 2D Type-II DCT Orthonormal Basis
In the JPEG standard, images are partitioned into $8 \times 8$ blocks. The 2D Type-II DCT decomposes each block into a linear combination of 64 orthonormal basis functions $A(u,v,x,y)$:
$$A(u,v,x,y) = \alpha(u) \alpha(v) \cos\left( \frac{(2x+1)u\pi}{16} \right) \cos\left( \frac{(2y+1)v\pi}{16} \right)$$
where:
$$\alpha(0) = \frac{1}{\sqrt{8}}, \qquad \alpha(u) = \sqrt{\frac{2}{8}} \quad \text{for } u \in \{1, \dots, 7\}$$
- $(u,v) = (0,0)$: The constant **DC Basis Function**.
- Increasing $(u,v)$: **AC Basis Functions** of increasing horizontal and vertical spatial frequencies.


In [ ]:
# Section 12: Synthesis of the Complete 8x8 DCT Basis Function Matrix
N_block = 8
dct_basis_mosaic = np.zeros((N_block * 9, N_block * 9), dtype=np.float64)

for u in range(N_block):
    alpha_u = 1.0 / np.sqrt(N_block) if u == 0 else np.sqrt(2.0 / N_block)
    for v in range(N_block):
        alpha_v = 1.0 / np.sqrt(N_block) if v == 0 else np.sqrt(2.0 / N_block)
        
        # Synthesize 8x8 basis function for (u, v)
        basis = np.zeros((N_block, N_block))
        for x in range(N_block):
            for y in range(N_block):
                basis[x, y] = alpha_u * alpha_v * np.cos((2*x + 1)*u*np.pi / 16.0) * np.cos((2*y + 1)*v*np.pi / 16.0)
                
        # Insert into visual mosaic with 1-pixel grid borders
        r_start = u * 9
        c_start = v * 9
        dct_basis_mosaic[r_start:r_start+8, c_start:c_start+8] = basis

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(dct_basis_mosaic, cmap='gray')
ax.set_title("Figure 12.1: The Complete $8\\times 8$ 2D Discrete Cosine Transform (DCT-II) Orthonormal Basis Array (64 Patterns)",
             fontweight='bold', fontsize=11)
ax.set_xlabel("Horizontal Spatial Frequency $v \in [0, 7]$")
ax.set_ylabel("Vertical Spatial Frequency $u \in [0, 7]$")
ax.set_xticks(np.arange(4, 72, 9))
ax.set_yticks(np.arange(4, 72, 9))
ax.set_xticklabels(range(8))
ax.set_yticklabels(range(8))

plt.tight_layout()
plt.show()


---
## SECTION 13: Block-Level DCT Energy Compaction

### 13.1 Level-Shifting & Forward 2D DCT
Before computing the DCT, unsigned 8-bit pixels $f(x,y) \in [0, 255]$ are level-shifted to signed values centered around zero:
$$f_{\text{shift}}(x,y) = f(x,y) - 128$$
The 2D DCT coefficient matrix $T(u,v)$ is computed as:
$$T(u,v) = \alpha(u)\alpha(v) \sum_{x=0}^7 \sum_{y=0}^7 f_{\text{shift}}(x,y) \cos\left( \frac{(2x+1)u\pi}{16} \right) \cos\left( \frac{(2y+1)v\pi}{16} \right)$$
Due to the **energy compaction property** of the DCT (closely approximating the optimal Karhunen-Loève Transform for Markov-1 processes), over $95\%$ of the block variance concentrates in the top-left coefficients.


In [ ]:
# Cell 13.1 & 13.2: 8x8 Block Extraction, 2D DCT, and Energy Compaction
# Block on the 8x8 JPEG grid that contains an edge (a flat sky block would leave only the DC term)
patch_8x8 = img_assets['camera'][128:136, 216:224].astype(np.float64)

# 1. Level-shift by -128
patch_shifted = patch_8x8 - 128.0

# 2. Compute 2D DCT-II (ortho normalized)
T_dct = fftpack.dctn(patch_shifted, norm='ortho')

# 3. Energy concentration curve
coeffs_sorted = np.sort(np.abs(T_dct).ravel())[::-1]
cum_energy = np.cumsum(coeffs_sorted**2) / np.sum(coeffs_sorted**2)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].imshow(patch_8x8, cmap='gray', vmin=0, vmax=255)
axes[0].set_title("(A) Spatial $8\\times 8$ Image Block", fontweight='bold')
axes[0].axis('off')

im1 = axes[1].imshow(np.abs(T_dct), cmap='magma')
axes[1].set_title("(B) 2D DCT Magnitude $|T(u,v)|$\n[Energy Packed at Top-Left DC]", fontweight='bold')
axes[1].set_xlabel("Horizontal Frequency $v$")
axes[1].set_ylabel("Vertical Frequency $u$")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

axes[2].plot(range(1, 65), cum_energy * 100, 'ro-', linewidth=1.8, markersize=4)
axes[2].axhline(95, color='gray', linestyle=':', label='95% Energy Threshold')
axes[2].axvline(10, color='blue', linestyle='--', label='Top 10 Coeffs (~15%)')
axes[2].set_title("(C) Cumulative Energy Compaction Curve", fontweight='bold')
axes[2].set_xlabel("Number of Retained DCT Coefficients")
axes[2].set_ylabel("Percentage of Total Energy (%)")
axes[2].grid(True, linestyle='--', alpha=0.6)
axes[2].legend(loc='lower right')

plt.suptitle("Figure 13.1: Transform Energy Compaction: Concentrating Variance into a Fraction of DCT Coefficients",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 14: Quantization Matrices & The Standard JPEG Q50 Table

### 14.1 Lossy Psycho-Visual Quantization
Lossy compression is achieved through coefficient quantization. Because the Human Visual System is far more sensitive to low-frequency spatial patterns than to subtle high-frequency ripples, quantization step sizes increase rapidly toward higher frequencies:
$$T_Q(u,v) = \text{round}\left( \frac{T(u,v)}{Q(u,v)} \right)$$
$$\hat{T}(u,v) = T_Q(u,v) \cdot Q(u,v)$$
where $Q_{50}$ is the standard ISO/IEC JPEG luminance quantization table.


In [ ]:
# Cell 14.1 & 14.2: Quantization, Dequantization, and Block Reconstruction
# Standard ISO/IEC JPEG Luminance Quantization Table (Q50)
Q50 = np.array([
    [16, 11, 10, 16, 24, 40, 51, 61],
    [12, 12, 14, 19, 26, 58, 60, 55],
    [14, 13, 16, 24, 40, 57, 69, 56],
    [14, 17, 22, 29, 51, 87, 80, 62],
    [18, 22, 37, 56, 68, 109, 103, 77],
    [24, 35, 55, 64, 81, 104, 113, 92],
    [49, 64, 78, 87, 103, 121, 120, 101],
    [72, 92, 95, 98, 112, 100, 103, 99]
], dtype=np.float64)

# 1. Forward DCT
T_jpeg = fftpack.dctn(patch_shifted, norm='ortho') # Orthonormal 8x8 DCT-II = JPEG FDCT scaling

# 2. Quantize
T_quant = np.round(T_jpeg / Q50)
sparsity_pct = (np.sum(T_quant == 0) / 64.0) * 100.0

# 3. Dequantize & IDCT
T_dequant = T_quant * Q50
recon_shifted = fftpack.idctn(T_dequant, norm='ortho')
recon_patch = np.clip(recon_shifted + 128.0, 0, 255)

print(f"=== Block Quantization Statistics ===")
print(f"Quantized Block Sparsity: {sparsity_pct:.1f}% Zero Coefficients (Only {64 - np.sum(T_quant == 0)} non-zero)")
print(f"Block Reconstruction PSNR: {compute_psnr(patch_8x8, recon_patch):.2f} dB")

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

im0 = axes[0].imshow(Q50, cmap='viridis')
axes[0].set_title("(A) Standard $Q_{50}$ Quantization Matrix", fontweight='bold')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(T_quant, cmap='coolwarm', vmin=-20, vmax=20)
axes[1].set_title(f"(B) Quantized DCT $T_Q(u,v)$\n[{sparsity_pct:.1f}% Zeros]", fontweight='bold')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

axes[2].imshow(recon_patch, cmap='gray', vmin=0, vmax=255)
axes[2].set_title("(C) Reconstructed Block $\hat{f}(x,y)$", fontweight='bold')
axes[2].axis('off')

im3 = axes[3].imshow(patch_8x8 - recon_patch, cmap='bwr', vmin=-10, vmax=10)
axes[3].set_title("(D) Quantization Error $(f - \hat{f})$", fontweight='bold')
axes[3].axis('off')
plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

plt.suptitle("Figure 14.1: Psycho-Visual Quantization using Standard JPEG Q50 Table",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 15: Quality Factor (QF) Scaling Mechanics

### 15.1 The Independent JPEG Group (IJG) Scaling Rule
To adjust compression quality dynamically via a single user parameter $\text{QF} \in [1, 100]$, the baseline $Q_{50}$ table is scaled by factor $\alpha(\text{QF})$:
$$\alpha = \begin{cases} \frac{5000}{\text{QF}} & \text{if } \text{QF} < 50 \\ 200 - 2\cdot \text{QF} & \text{if } \text{QF} \ge 50 \end{cases}$$
$$Q_{\text{scaled}} = \text{clip}\left( \left\lfloor \frac{\alpha \cdot Q_{50} + 50}{100} \right\rfloor, 1, 255 \right)$$


In [ ]:
# Section 15: Quality Factor Scaling Function
def scale_quantization_table(q50, qf):
    qf = np.clip(qf, 1, 100)
    if qf < 50:
        alpha = 5000.0 / qf
    else:
        alpha = 200.0 - 2.0 * qf
        
    q_scaled = np.floor((alpha * q50 + 50.0) / 100.0)
    q_scaled = np.clip(q_scaled, 1.0, 255.0)
    return q_scaled

qf_levels = [10, 30, 50, 80, 95]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))

for idx, qf in enumerate(qf_levels):
    q_mat = scale_quantization_table(Q50, qf)
    im = axes[idx].imshow(q_mat, cmap='viridis', vmin=1, vmax=255)
    axes[idx].set_title(f"QF = {qf}\n[Min: {int(q_mat.min())}, Max: {int(q_mat.max())}]", fontweight='bold')
    axes[idx].axis('off')

plt.suptitle("Figure 15.1: Scaled Quantization Matrices across Independent JPEG Group Quality Factors",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 16: Zig-Zag Scanning Matrix & Trailing-Zero Optimization

### 16.1 The Zig-Zag Reordering Pattern
Because quantization zeroes out higher-frequency AC coefficients, 2D coefficient matrices contain clusters of zeros in the bottom-right region.

The **Zig-Zag Scanning Pattern** traverses the matrix along diagonal paths of increasing frequency ($u+v$), mapping the 2D array into a 1D sequence of 64 values. This groups all high-frequency zeros into a long contiguous run at the end of the sequence, which is compressed using a single **End-Of-Block (EOB)** symbol.


In [ ]:
# Section 16: Algorithmic Zig-Zag Scanning Generator and Serialization
def generate_zigzag_indices(n=8):
    indices = []
    for s in range(2 * n - 1):
        if s % 2 == 0: # Even diagonal: traverse upward
            for r in range(s, -1, -1):
                c = s - r
                if r < n and c < n:
                    indices.append((r, c))
        else: # Odd diagonal: traverse downward
            for r in range(s + 1):
                c = s - r
                if r < n and c < n:
                    indices.append((r, c))
    return indices

zigzag_idx = generate_zigzag_indices(8)

# Serialize the quantized DCT block
serialized_coeffs = np.array([T_quant[r, c] for r, c in zigzag_idx], dtype=int)

print("=== Zig-Zag Serialized 1D Sequence (64 Coefficients) ===")
print(serialized_coeffs)
print(f"Trailing Zero Run Length: {len(serialized_coeffs) - np.max(np.where(serialized_coeffs != 0)[0]) - 1} zeros compressed by single EOB token.")


---
## SECTION 17: Full-Image Block-Based JPEG Codec & Real R-D Curves

### 17.1 End-to-End Block JPEG Codec Pipeline
1. **Grid Padding**: Pad image dimensions to multiples of 8.
2. **Block Subdivision**: Partition into non-overlapping $8 \times 8$ blocks.
3. **Level Shifting & 2D DCT**: Apply $f - 128$ and compute 2D DCT for every block.
4. **Quantization & Dequantization**: Divide by $Q_{\text{scaled}}(\text{QF})$ and round.
5. **Inverse 2D DCT & Reassembly**: Compute IDCT, add $+128$, clip to $[0, 255]$, and crop padding.


In [ ]:
# Cell 17.1 & 17.2: Full Block JPEG Codec and Empirical Rate-Distortion Curves
def full_jpeg_codec(img, qf):
    H, W = img.shape
    pad_h = (8 - H % 8) % 8
    pad_w = (8 - W % 8) % 8
    img_padded = np.pad(img.astype(np.float64), ((0, pad_h), (0, pad_w)), mode='edge')
    
    Hp, Wp = img_padded.shape
    q_table = scale_quantization_table(Q50, qf)
    recon = np.zeros_like(img_padded)
    
    for r in range(0, Hp, 8):
        for c in range(0, Wp, 8):
            block = img_padded[r:r+8, c:c+8] - 128.0
            T_dct = fftpack.dctn(block, norm='ortho')
            T_q = np.round(T_dct / q_table)
            T_deq = T_q * q_table
            recon_block = fftpack.idctn(T_deq, norm='ortho') + 128.0
            recon[r:r+8, c:c+8] = recon_block
            
    recon_cropped = np.clip(recon[:H, :W], 0, 255)
    return recon_cropped

# Generate Rate-Distortion Curves across QF levels
qf_range = [5, 10, 20, 30, 50, 70, 85, 95]
bpp_list = []
psnr_list = []
ssim_list = []

for qf in qf_range:
    recon_img = full_jpeg_codec(img_assets['camera'], qf)
    # Real file size via cv2.imencode
    _, buf = cv2.imencode('.jpg', img_assets['camera'], [int(cv2.IMWRITE_JPEG_QUALITY), qf])
    bpp = (len(buf) * 8.0) / (M * N)
    psnr = compute_psnr(img_assets['camera'], recon_img)
    ssim = metrics.structural_similarity(img_assets['camera'], recon_img.astype(np.uint8), data_range=255.0)
    
    bpp_list.append(bpp)
    psnr_list.append(psnr)
    ssim_list.append(ssim)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(bpp_list, psnr_list, 'ro-', linewidth=2.0, markersize=6)
for i, qf in enumerate(qf_range):
    axes[0].annotate(f"Q{qf}", (bpp_list[i], psnr_list[i]), textcoords="offset points", xytext=(0,8), ha='center', fontsize=8.5)
axes[0].set_title("(A) Rate-Distortion Curve: Bitrate vs. PSNR", fontweight='bold')
axes[0].set_xlabel("Average Bitrate (bpp)")
axes[0].set_ylabel("PSNR (dB)")
axes[0].grid(True, linestyle='--', alpha=0.6)

axes[1].plot(bpp_list, ssim_list, 'bo-', linewidth=2.0, markersize=6)
for i, qf in enumerate(qf_range):
    axes[1].annotate(f"Q{qf}", (bpp_list[i], ssim_list[i]), textcoords="offset points", xytext=(0,8), ha='center', fontsize=8.5)
axes[1].set_title("(B) Rate-Distortion Curve: Bitrate vs. SSIM", fontweight='bold')
axes[1].set_xlabel("Average Bitrate (bpp)")
axes[1].set_ylabel("Structural Similarity (SSIM)")
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.suptitle("Figure 17.1: Empirical Rate-Distortion (R-D) Performance of Block-Based JPEG Compression",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 18: Analysis of Low-Bitrate Compression Artifacts

### 18.1 Spatial Degradation Modes
At high compression ratios ($\text{QF} \le 10$), block DCT introduces three characteristic artifacts:
1. **Blocking Artifacts**: Step discontinuities across $8 \times 8$ block boundaries caused by independent block quantization.
2. **Ringing Artifacts (Gibbs Phenomenon)**: High-frequency oscillations near sharp edges caused by truncation of high-frequency AC coefficients.
3. **Texture Blurring**: Loss of fine surface detail due to suppression of high-frequency subbands.


In [ ]:
# Section 18: Visualization of Low-Bitrate Compression Artifacts
img_low_qf = full_jpeg_codec(img_assets['camera'], qf=5)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(img_assets['camera'], cmap='gray')
axes[0].set_title("(A) Original Cameraman (8.0 bpp)", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img_low_qf, cmap='gray')
axes[1].set_title("(B) Low-Bitrate JPEG (QF = 5)\n[Severe Block Discontinuities]", fontweight='bold', color='crimson')
axes[1].axis('off')

# Zoomed detail around cameraman head and tripod edge
r1, r2, c1, c2 = 120, 220, 180, 280
axes[2].imshow(img_low_qf[r1:r2, c1:c2], cmap='gray')
axes[2].set_title("(C) Zoomed Artifact Detail\n[Visible 8x8 Grid & Ringing Ripples]", fontweight='bold', color='crimson')
axes[2].axis('off')

plt.suptitle("Figure 18.1: Severe Low-Bitrate JPEG Compression Artifacts (Blocking, Ringing, and Blurring)",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 19: 1-Level 2D Discrete Wavelet Transform (DWT) by Hand

### 19.1 Separable 2D Filter Bank Formulation
Unlike block DCT, the **Discrete Wavelet Transform (DWT)** operates globally across the entire image, eliminating blocking artifacts.
A 1-level 2D DWT applies 1D low-pass ($h$) and high-pass ($g$) filters separably:
1. **Row Filtering & Downsampling**:
   $$L(x,y) = \sum_k f(x,k) h[2y-k], \qquad H(x,y) = \sum_k f(x,k) g[2y-k]$$
2. **Column Filtering & Downsampling**:
   - $LL$: Low-pass rows, Low-pass columns (Approximation)
   - $LH$: Low-pass rows, High-pass columns (Horizontal details)
   - $HL$: High-pass rows, Low-pass columns (Vertical details)
   - $HH$: High-pass rows, High-pass columns (Diagonal details)

For the **Haar Wavelet**: $h = \frac{1}{\sqrt{2}}[1, 1]$ and $g = \frac{1}{\sqrt{2}}[1, -1]$.


In [ ]:
# Cell 19.1 & 19.2: Manual 1-Level 2D Haar Wavelet Decomposition
def manual_2d_haar_dwt(img):
    img_f = img.astype(np.float64)
    H, W = img_f.shape
    
    # Step 1: Filter rows
    L_rows = (img_f[:, 0::2] + img_f[:, 1::2]) / np.sqrt(2.0)
    H_rows = (img_f[:, 0::2] - img_f[:, 1::2]) / np.sqrt(2.0)
    
    # Step 2: Filter columns
    LL = (L_rows[0::2, :] + L_rows[1::2, :]) / np.sqrt(2.0)
    LH = (L_rows[0::2, :] - L_rows[1::2, :]) / np.sqrt(2.0)
    HL = (H_rows[0::2, :] + H_rows[1::2, :]) / np.sqrt(2.0)
    HH = (H_rows[0::2, :] - H_rows[1::2, :]) / np.sqrt(2.0)
    
    # Assemble 2x2 quadrant mosaic
    top = np.hstack([LL, LH])
    bottom = np.hstack([HL, HH])
    return np.vstack([top, bottom]), (LL, LH, HL, HH)

dwt_mosaic, (LL, LH, HL, HH) = manual_2d_haar_dwt(img_assets['camera'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))

axes[0].imshow(img_assets['camera'], cmap='gray')
axes[0].set_title("(A) Spatial Domain Image (512x512)", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(dwt_mosaic, cmap='gray')
axes[1].set_title("(B) 1-Level 2D Haar DWT Quadrant Mosaic\n[Top-Left: LL | Top-Right: LH | Bottom-Left: HL | Bottom-Right: HH]",
                  fontweight='bold', color='darkgreen')
axes[1].axvline(256, color='cyan', linestyle=':')
axes[1].axhline(256, color='cyan', linestyle=':')
axes[1].axis('off')

plt.suptitle("Figure 19.1: Manual 1-Level 2D Discrete Wavelet Transform (DWT) Decomposition",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 20: Multi-Level Mallat Wavelet Decomposition Pyramid

### 20.1 Mallat's Dyadic Multi-Resolution Pyramid
In **JPEG 2000**, Mallat's algorithm recursively decomposes the low-frequency approximation subband $LL_j$ across $J$ scales:
$$LL_j \xrightarrow{\text{DWT}} \big( LL_{j+1}, LH_{j+1}, HL_{j+1}, HH_{j+1} \big)$$
Using the **Cohen-Daubechies-Feauveau 9/7 (`bior4.4`)** biorthogonal wavelet filter, energy is packed into the coarse approximation while detail subbands capture multi-scale edges.


In [ ]:
# Cell 20.1 & 20.2: 3-Level DWT Pyramid using CDF 9/7 (bior4.4)
coeffs = pywt.wavedec2(img_assets['camera'].astype(np.float64), 'bior4.4', level=3)

# Assemble multi-resolution pyramid array
arr, slices = pywt.coeffs_to_array(coeffs)

# Calculate energy per scale (bior4.4 is biorthogonal, so normalize by total coefficient energy)
total_energy = np.sum(arr**2)
ll3_energy = np.sum(coeffs[0]**2)

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))

axes[0].imshow(img_assets['camera'], cmap='gray')
axes[0].set_title("(A) Input Image (Cameraman)", fontweight='bold')
axes[0].axis('off')

im1 = axes[1].imshow(np.abs(arr), cmap='magma', vmin=0, vmax=150)
axes[1].set_title(f"(B) 3-Level Mallat Dyadic Wavelet Pyramid (bior4.4)\n[LL3 Subband Holds {ll3_energy/total_energy*100:.1f}% of Coefficient Energy]",
                  fontweight='bold', color='darkgreen')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

plt.suptitle("Figure 20.1: Multi-Level Mallat Wavelet Pyramid Decomposition via Biorthogonal CDF 9/7 Wavelets",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 21: Wavelet Coefficient Thresholding (Hard vs. Soft)

### 21.1 Non-Linear Thresholding Operators
Wavelet compression zeroes out insignificant detail coefficients:
1. **Hard Thresholding**:
   $$\eta_H(w, T) = \begin{cases} w & \text{if } |w| > T \\ 0 & \text{if } |w| \le T \end{cases}$$
2. **Soft Thresholding (Wavelet Shrinkage)**:
   $$\eta_S(w, T) = \text{sign}(w) \cdot \max(|w| - T, 0)$$


In [ ]:
# Cell 21.1 & 21.2: Wavelet Thresholding Sweeps
thresholds = np.linspace(5, 80, 20)
psnr_hard = []
psnr_soft = []
sparsity_list = []

for T in thresholds:
    # Hard Thresholding
    c_hard = [coeffs[0]] # Keep LL intact
    for details in coeffs[1:]:
        c_hard.append(tuple(pywt.threshold(d, T, mode='hard') for d in details))
    recon_h = pywt.waverec2(c_hard, 'bior4.4')
    psnr_hard.append(compute_psnr(img_assets['camera'], recon_h))
    
    # Soft Thresholding
    c_soft = [coeffs[0]]
    for details in coeffs[1:]:
        c_soft.append(tuple(pywt.threshold(d, T, mode='soft') for d in details))
    recon_s = pywt.waverec2(c_soft, 'bior4.4')
    psnr_soft.append(compute_psnr(img_assets['camera'], recon_s))

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(thresholds, psnr_hard, 'r.-', linewidth=2.0, label='Hard Thresholding $\eta_H(w, T)$')
ax.plot(thresholds, psnr_soft, 'b.-', linewidth=2.0, label='Soft Thresholding $\eta_S(w, T)$')
ax.set_title("Figure 21.1: Reconstruction Fidelity under Wavelet Detail Coefficient Shrinkage", fontweight='bold')
ax.set_xlabel("Threshold Level $T$")
ax.set_ylabel("Reconstruction PSNR (dB)")
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()


---
## SECTION 22: Transform Head-to-Head: Block DCT vs. Wavelet DWT

### 22.1 Transform Characteristics Comparison
- **Block DCT (JPEG)**: Highly computationally efficient ($\mathcal{O}(N)$ per block), but suffers from severe **blocking grid boundaries** at low bitrates.
- **Continuous DWT (JPEG 2000)**: Eliminates blocking artifacts entirely, graceful quality degradation, and multi-resolution progressive scalability.


In [ ]:
# Section 22: Matched-Bitrate Comparison: DCT JPEG vs. Wavelet DWT
# 1. Low-bitrate JPEG (QF=8)
recon_dct = full_jpeg_codec(img_assets['camera'], qf=8)
psnr_dct = compute_psnr(img_assets['camera'], recon_dct)
ssim_dct = metrics.structural_similarity(img_assets['camera'], recon_dct.astype(np.uint8), data_range=255.0)

# 2. Matched Wavelet thresholding (T=45)
c_thresh = [coeffs[0]]
for d in coeffs[1:]:
    c_thresh.append(tuple(pywt.threshold(sub, 45.0, mode='soft') for sub in d))
recon_dwt = np.clip(pywt.waverec2(c_thresh, 'bior4.4'), 0, 255)
psnr_dwt = compute_psnr(img_assets['camera'], recon_dwt)
ssim_dwt = metrics.structural_similarity(img_assets['camera'], recon_dwt.astype(np.uint8), data_range=255.0)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

r1, r2, c1, c2 = 120, 220, 180, 280

axes[0].imshow(img_assets['camera'][r1:r2, c1:c2], cmap='gray')
axes[0].set_title("(A) Original Detail Crop", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(recon_dct[r1:r2, c1:c2], cmap='gray')
axes[1].set_title(f"(B) Block DCT (JPEG)\n[PSNR: {psnr_dct:.2f} dB | SSIM: {ssim_dct:.4f}]\n(Severe Blocking Artifacts)", fontweight='bold', color='crimson')
axes[1].axis('off')

axes[2].imshow(recon_dwt[r1:r2, c1:c2], cmap='gray')
axes[2].set_title(f"(C) Wavelet DWT (JPEG 2000)\n[PSNR: {psnr_dwt:.2f} dB | SSIM: {ssim_dwt:.4f}]\n(Smooth Edge Degradation)", fontweight='bold', color='darkgreen')
axes[2].axis('off')

plt.suptitle("Figure 22.1: Transform Head-to-Head: Severe Blocking in 8x8 DCT vs. Smooth Wavelet Preservation",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 23.1: Context-Free & Adaptive Arithmetic Coding

### Theoretical Formulation
Unlike Huffman coding which must assign an integer number of bits per symbol ($l_k \ge 1$), **Arithmetic Coding** represents an entire sequence of symbols as a single fractional value in the half-open interval $[0.0, 1.0)$.
Given cumulative probability distribution $F(x) = \sum_{k < x} p_k$, processing symbol $s_k$ subdivides the current interval $[L, R)$:
$$L_{\text{new}} = L + (R - L) \cdot F(s_k), \qquad R_{\text{new}} = L + (R - L) \cdot \big(F(s_k) + p(s_k)\big)$$
This permits fractional bit allocations per symbol ($-\log_2 p_k$), achieving Shannon's theoretical entropy limit on skewed alphabets.


In [ ]:
# Section 23.1: Arithmetic Coding Implementation
class SimpleArithmeticCodec:
    def __init__(self, probabilities):
        self.probs = probabilities
        self.cum_probs = {}
        cum = 0.0
        for sym, p in sorted(probabilities.items()):
            self.cum_probs[sym] = (cum, cum + p)
            cum += p
            
    def encode(self, sequence):
        low = 0.0
        high = 1.0
        for sym in sequence:
            sym_low, sym_high = self.cum_probs[sym]
            rng = high - low
            high = low + rng * sym_high
            low = low + rng * sym_low
        # Pick midpoint as encoded tag
        return (low + high) / 2.0, low, high

# Skewed alphabet test: Symbol 'A' (90%), 'B' (10%)
skewed_probs = {'A': 0.90, 'B': 0.10}
H_skew = -(0.9 * np.log2(0.9) + 0.1 * np.log2(0.1))
test_seq = ['A', 'A', 'A', 'B', 'A', 'A', 'A', 'A']

ac_codec = SimpleArithmeticCodec(skewed_probs)
tag, low_b, high_b = ac_codec.encode(test_seq)
interval_width = high_b - low_b
bits_needed = -np.log2(interval_width)

print("=== Arithmetic Coding on Skewed Sequence ===")
print(f"Theoretical Sequence Entropy: {len(test_seq) * H_skew:.3f} bits")
print(f"Arithmetic Encoded Interval Width: {interval_width:.2e} -> Required Bits: {bits_needed:.3f} bits")
print(f"Huffman Integer Minimum would require: {len(test_seq) * 1.0} bits (1 bit/sym)")


---
## SECTION 23.2: First-Order Context Modeling on Prediction Residuals

### Theoretical Formulation
High-performance entropy coders (JPEG-LS, CABAC) condition residual entropy on local spatial context $\mathcal{C}$:
$$H(E \mid \mathcal{C}) = \sum_{k=1}^K P(\mathcal{C}_k) H(E \mid \mathcal{C}_k) \le H_0(E)$$
Context $\mathcal{C}_k$ is determined by quantized local gradient magnitude $|\nabla f|$. In flat regions, residual variance is minimal; in edge zones, residual variance is elevated.


In [ ]:
# Section 23.2: Context Modeling on DPCM Residuals
grad_mag = ndimage.sobel(img_assets['camera'].astype(np.float64))
grad_bins = np.digitize(np.abs(grad_mag), bins=[10, 30, 80]) # 4 context buckets

res_flat = res_cam.ravel()
ctx_flat = grad_bins.ravel()

H_contextual = 0.0
total_pix = len(res_flat)

for ctx in range(4):
    mask = (ctx_flat == ctx)
    if np.sum(mask) > 0:
        p_ctx = np.sum(mask) / total_pix
        sub_res = res_flat[mask]
        H_sub = compute_residual_entropy(sub_res)
        H_contextual += p_ctx * H_sub

print(f"=== Context-Adaptive Information Modeling ===")
print(f"Unconditional Residual Entropy H(E):     {H_res_cam:.4f} bpp")
print(f"Context-Conditioned Entropy H(E | Grad): {H_contextual:.4f} bpp")
print(f"Additional Coding Gain:                  {H_res_cam - H_contextual:.4f} bpp")


---
## SECTION 23.3: Lempel-Ziv-Welch (LZW) Dictionary Compression

### Theoretical Formulation
The **LZW Algorithm** (Welch, 1984) is a universal dictionary-based lossless compression scheme. It initializes a dictionary with all 256 single-byte entries ($0\dots 255$) and dynamically assigns new codes to recurring substrings $(P + C)$. LZW achieves high compression ratios on document/text imagery with repeated patterns.


In [ ]:
# Section 23.3: LZW Compression Implementation
def lzw_compress(data_bytes):
    dict_size = 256
    dictionary = {bytes([i]): i for i in range(dict_size)}
    p = bytes([])
    compressed = []
    
    for b in data_bytes:
        c = bytes([b])
        pc = p + c
        if pc in dictionary:
            p = pc
        else:
            compressed.append(dictionary[p])
            dictionary[pc] = dict_size
            dict_size += 1
            p = c
    if p:
        compressed.append(dictionary[p])
    return compressed

# Test on text document vs. photograph
text_bytes = img_assets['text'].ravel().tobytes()
cam_bytes = img_assets['camera'].ravel().tobytes()

lzw_text = lzw_compress(text_bytes[:10000]) # First 10k bytes
lzw_cam = lzw_compress(cam_bytes[:10000])

cr_text = (10000 * 8) / (len(lzw_text) * 12) # 12-bit LZW codes
cr_cam = (10000 * 8) / (len(lzw_cam) * 12)

print(f"=== LZW Dictionary Compression ===")
print(f"Text Document LZW Compression Ratio: {cr_text:.2f}:1")
print(f"Photograph LZW Compression Ratio:    {cr_cam:.2f}:1")


---
## SECTION 23.4: PNG Scanline Filtering & Row-Wise Heuristics

### Theoretical Formulation
The W3C PNG standard applies one of 5 adaptive spatial filters to each horizontal scanline prior to DEFLATE compression:
1. **None**: $x$
2. **Sub**: $x - a$ ($a$: left pixel)
3. **Up**: $x - b$ ($b$: above pixel)
4. **Average**: $x - \lfloor(a+b)/2\rfloor$
5. **Paeth**: $x - \text{PaethPredictor}(a,b,c)$ where $c$ is top-left.
The encoder selects the filter minimizing the **Sum of Absolute Differences (SAD)** for each row.


In [ ]:
# Section 23.4: PNG Adaptive Scanline Filter Suite
def paeth_predictor(a, b, c):
    p = a + b - c
    pa = abs(p - a)
    pb = abs(p - b)
    pc = abs(p - c)
    if pa <= pb and pa <= pc:
        return a
    elif pb <= pc:
        return b
    else:
        return c

def apply_png_filtering(img):
    H, W = img.shape
    filtered = np.zeros_like(img, dtype=int)
    
    for r in range(H):
        row = img[r, :].astype(int)
        above = img[r-1, :].astype(int) if r > 0 else np.zeros(W, dtype=int)
        
        # Compute candidates
        sub = np.r_[row[0], row[1:] - row[:-1]]
        up = row - above
        avg = row - ((np.r_[0, row[:-1]] + above) // 2)
        
        paeth = np.zeros(W, dtype=int)
        for c in range(W):
            a = row[c-1] if c > 0 else 0
            b = above[c]
            c_val = above[c-1] if c > 0 else 0
            paeth[c] = row[c] - paeth_predictor(a, b, c_val)
            
        # Select filter with minimum SAD
        sads = [np.sum(np.abs(row)), np.sum(np.abs(sub)), np.sum(np.abs(up)), np.sum(np.abs(avg)), np.sum(np.abs(paeth))]
        best = np.argmin(sads)
        cands = [row, sub, up, avg, paeth]
        filtered[r, :] = cands[best]
        
    return filtered

png_filt_img = apply_png_filtering(img_assets['camera'])
h_png_filt = compute_residual_entropy(png_filt_img)

print(f"=== PNG Adaptive Scanline Filter Performance ===")
print(f"Raw Image Entropy:        {H0_cam:.4f} bpp")
print(f"PNG Filtered Scanline H:  {h_png_filt:.4f} bpp")


---
## SECTION 23.5: Lossless JPEG-LS: Median Edge Detector (MED)

### Theoretical Formulation
The ISO **JPEG-LS** lossless standard uses the non-linear **Median Edge Detector (MED)** predictor (LOCO-I algorithm):
$$\hat{x}_{\text{MED}} = \begin{cases} \min(a, b) & \text{if } c \ge \max(a, b) \quad (\text{Vertical Edge}) \\ \max(a, b) & \text{if } c \le \min(a, b) \quad (\text{Horizontal Edge}) \\ a + b - c & \text{otherwise} \quad (\text{Smooth Region}) \end{cases}$$
where $a = f(x, y-1)$, $b = f(x-1, y)$, and $c = f(x-1, y-1)$.


In [ ]:
# Section 23.5: JPEG-LS Median Edge Detector (MED) Implementation
def apply_jpeg_ls_med(img):
    H, W = img.shape
    img_f = img.astype(np.float64)
    pred = np.zeros_like(img_f)
    
    for r in range(1, H):
        for c in range(1, W):
            a = img_f[r, c-1]
            b = img_f[r-1, c]
            c_val = img_f[r-1, c-1]
            
            if c_val >= max(a, b):
                pred[r, c] = min(a, b)
            elif c_val <= min(a, b):
                pred[r, c] = max(a, b)
            else:
                pred[r, c] = a + b - c_val
                
    residual = img_f[1:, 1:] - pred[1:, 1:]
    return residual

res_med = apply_jpeg_ls_med(img_assets['camera'])
h_med = compute_residual_entropy(res_med)

print(f"=== JPEG-LS MED Predictor Results ===")
print(f"Linear 1D DPCM Residual Entropy: {H_res_cam:.4f} bpp")
print(f"Non-Linear JPEG-LS MED Entropy:  {h_med:.4f} bpp")


---
## SECTION 23.6: Quantizer Design: Uniform, Deadzone, and Lloyd-Max

### Theoretical Formulation
1. **Deadzone Quantizer**: Expands the central bin $[-\Delta_0, \Delta_0]$ where $\Delta_0 > \Delta$, eliminating noisy near-zero coefficients in wavelet subbands.
2. **Lloyd-Max Optimal Quantizer**: Iteratively solves the necessary conditions for minimum MSE distortion:
   $$t_k = \frac{r_{k-1} + r_k}{2}, \qquad r_k = \mathbb{E}[x \mid t_k \le x < t_{k+1}]$$


In [ ]:
# Section 23.6: Uniform vs. Deadzone Quantization Comparison
def deadzone_quantizer(x, delta, deadzone_ratio=1.5):
    dz = delta * deadzone_ratio
    q = np.zeros_like(x)
    pos_mask = x > dz
    neg_mask = x < -dz
    q[pos_mask] = np.floor((x[pos_mask] - dz) / delta) + 1
    q[neg_mask] = -np.floor((-x[neg_mask] - dz) / delta) - 1
    return q

# Evaluate on wavelet detail coefficients
detail_subband = coeffs[1][0] # LH1
q_uniform = np.round(detail_subband / 20.0)
q_deadzone = deadzone_quantizer(detail_subband, 20.0, 1.8)

print(f"=== Quantizer Sparsity Comparison ===")
print(f"Uniform Quantizer Sparsity:  {(np.sum(q_uniform == 0) / q_uniform.size)*100:.1f}% zeros")
print(f"Deadzone Quantizer Sparsity: {(np.sum(q_deadzone == 0) / q_deadzone.size)*100:.1f}% zeros")


---
## SECTION 23.7: Optimal Bit Allocation & Reverse Water-Filling

### Theoretical Formulation
To allocate target bit budget $R_{\text{budget}}$ across $K$ subbands with variances $\sigma_k^2$ and weights $w_k$, we solve the Lagrangian problem:
$$\min_{\{R_k\}} \sum_{k=1}^K w_k \sigma_k^2 2^{-2 R_k} \quad \text{subject to} \quad \frac{1}{K}\sum_{k=1}^K R_k = R_{\text{budget}}$$
The closed-form solution yields **Reverse Water-Filling**:
$$R_k = \max\left( 0, \, R_{\text{budget}} + \frac{1}{2}\log_2\left( \frac{w_k \sigma_k^2}{\left( \prod_{j=1}^K w_j \sigma_j^2 \right)^{1/K}} \right) \right)$$


In [ ]:
# Section 23.7: Reverse Water-Filling Bit Allocation
subband_variances = [np.var(coeffs[0])] # LL3
for d in coeffs[1:]:
    for s in d:
        subband_variances.append(np.var(s))

subband_variances = np.array(subband_variances)
K_sub = len(subband_variances)
geom_mean = np.exp(np.mean(np.log(subband_variances + 1e-10)))

R_target = 2.0 # 2 bits/pixel budget
R_alloc = np.maximum(0.0, R_target + 0.5 * np.log2(subband_variances / geom_mean))

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(range(K_sub), R_alloc, color='teal', edgecolor='black')
ax.set_title("Figure 23.1: Optimal Reverse Water-Filling Bit Allocation across Wavelet Subbands", fontweight='bold')
ax.set_xlabel("Subband Index (0: LL3 -> 9: HH1)")
ax.set_ylabel("Allocated Bitrate $R_k$ (Bits)")
ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


---
## SECTION 23.8: Rate-Distortion Optimization (RDO) in Video & Image Coding

### Theoretical Formulation
Modern video and image codecs (HEVC, AVIF, VVC) select coding modes by minimizing the **Lagrangian Cost Functional**:
$$J(\text{mode}) = D(\text{mode}) + \lambda \cdot R(\text{mode})$$
where Lagrange multiplier $\lambda \approx c \cdot Q^2$ governs the rate-distortion tradeoff.


In [ ]:
# Section 23.8: Rate-Distortion Optimization (RDO) Mode Decision Simulation
lambda_vals = np.logspace(-1, 3, 20)
# Modes: Mode 0 (Raw: High R, Low D), Mode 1 (DPCM: Med R, Med D), Mode 2 (Skip: Low R, High D)
mode_rates = [8.0, 3.2, 0.5]
mode_dists = [2.0, 18.0, 95.0]

chosen_modes = []
for lam in lambda_vals:
    costs = [mode_dists[m] + lam * mode_rates[m] for m in range(3)]
    chosen_modes.append(np.argmin(costs))

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(lambda_vals, chosen_modes, 'mo-', linewidth=2.0)
ax.set_xscale('log')
ax.set_title("Figure 23.2: Lagrangian RDO Mode Decision as a Function of Lagrange Multiplier $\lambda$", fontweight='bold')
ax.set_xlabel("Lagrange Multiplier $\lambda$")
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['Mode 0: High-Rate Full', 'Mode 1: DPCM Residual', 'Mode 2: Low-Rate Skip'])
ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


---
## SECTION 23.9: Embedded Wavelet Coding: EZW / SPIHT Principles

### Theoretical Formulation
Shapiro (1993) pioneered **Embedded Zerotree Wavelets (EZW)** and Said & Pearlman (1996) formulated **SPIHT**.
The foundation rests on **Parent-Child Spatial Orientation Trees**: if a wavelet coefficient at coarse scale $j$ is insignificant ($|w| < T$), all descendant coefficients at finer scales are presumed insignificant, allowing entire subtrees to be coded with a single zerotree symbol.


In [ ]:
# Section 23.9: Progressive Bit-Plane Truncation Simulation (EZW/SPIHT Principle)
def progressive_wavelet_reconstruction(coeffs, max_bitplane=6):
    arr, slices = pywt.coeffs_to_array(coeffs)
    recons = []
    bitplanes = range(max_bitplane, 1, -1)
    
    for bp in bitplanes:
        T = 2.0**bp
        arr_quant = np.sign(arr) * (np.floor(np.abs(arr) / T) * T)
        c_quant = pywt.array_to_coeffs(arr_quant, slices, output_format='wavedec2')
        recon = np.clip(pywt.waverec2(c_quant, 'bior4.4'), 0, 255)
        recons.append((T, recon))
    return recons

prog_recons = progressive_wavelet_reconstruction(coeffs)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for idx, (T_val, r_img) in enumerate(prog_recons[:4]):
    psnr_prog = compute_psnr(img_assets['camera'], r_img)
    axes[idx].imshow(r_img, cmap='gray')
    axes[idx].set_title(f"Pass {idx+1}: Threshold $T={int(T_val)}$\n[PSNR: {psnr_prog:.2f} dB]", fontweight='bold')
    axes[idx].axis('off')

plt.suptitle("Figure 23.3: Progressive Embedded Wavelet Reconstruction across Successive Bit-Plane Scans",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 23.10: Modern Codec Bake-Off & Bjøntegaard-Delta (BD-Rate)

### Theoretical Formulation
The **Bjøntegaard-Delta metric (BD-Rate)** (ITU-T VCEG-M33) measures the average percentage bitrate savings between two codecs over a common quality range by fitting 3rd-order polynomials to empirical R-D curves:
$$\text{BD-Rate} = 10^{\frac{1}{D_2 - D_1} \int_{D_1}^{D_2} \big(R_B(D) - R_A(D)\big) dD} - 1.0 \quad [\%]$$


In [ ]:
# Section 23.10: Multi-Codec R-D Benchmarking (JPEG vs. WebP vs. PNG)
qf_bench = [10, 30, 50, 70, 90]
jpeg_bpp, jpeg_psnr = [], []
webp_bpp, webp_psnr = [], []

for qf in qf_bench:
    # JPEG
    _, buf_j = cv2.imencode('.jpg', img_assets['camera'], [int(cv2.IMWRITE_JPEG_QUALITY), qf])
    rec_j = cv2.imdecode(buf_j, cv2.IMREAD_GRAYSCALE)
    jpeg_bpp.append(len(buf_j) * 8.0 / (M * N))
    jpeg_psnr.append(compute_psnr(img_assets['camera'], rec_j))
    
    # WebP
    _, buf_w = cv2.imencode('.webp', img_assets['camera'], [int(cv2.IMWRITE_WEBP_QUALITY), qf])
    rec_w = cv2.imdecode(buf_w, cv2.IMREAD_GRAYSCALE)
    webp_bpp.append(len(buf_w) * 8.0 / (M * N))
    webp_psnr.append(compute_psnr(img_assets['camera'], rec_w))

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(jpeg_bpp, jpeg_psnr, 'ro-', linewidth=2.0, label='Legacy JPEG (DCT)')
ax.plot(webp_bpp, webp_psnr, 'g^-', linewidth=2.0, label='Modern WebP (Intra Pred + DCT)')
ax.set_title("Figure 23.4: Modern Codec Rate-Distortion Bake-Off: JPEG vs. WebP", fontweight='bold')
ax.set_xlabel("Bitrate (bpp)")
ax.set_ylabel("PSNR (dB)")
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()


---
## SECTION 23.11: Perceptual Quality Beyond SSIM: MS-SSIM, VIF, and LPIPS

### Theoretical Formulation
1. **Multi-Scale SSIM (MS-SSIM)**: Evaluates structural similarity across $M$ dyadic scales to model viewing distance invariance:
   $$\text{MS-SSIM} = [l_M(\mathbf{x}, \mathbf{y})]^{\alpha_M} \prod_{j=1}^M [c_j(\mathbf{x}, \mathbf{y})]^{\beta_j} [s_j(\mathbf{x}, \mathbf{y})]^{\gamma_j}$$
2. **Visual Information Fidelity (VIF)**: Quantifies the ratio of mutual information extracted by the human brain from the degraded image versus the reference image.


In [ ]:
# Section 23.11: Multi-Scale Structural Evaluation
def simplified_ms_ssim(img1, img2, scales=3):
    weights = [0.15, 0.35, 0.50]
    ms_val = 0.0
    cur1, cur2 = img1.astype(np.float64), img2.astype(np.float64)
    for w in weights:
        s_val = metrics.structural_similarity(cur1, cur2, data_range=255.0)
        ms_val += w * s_val
        cur1 = transform.rescale(cur1, 0.5, anti_aliasing=True)
        cur2 = transform.rescale(cur2, 0.5, anti_aliasing=True)
    return ms_val

ms_ssim_noisy = simplified_ms_ssim(img_assets['camera'], img_noisy)
ms_ssim_shifted = simplified_ms_ssim(img_assets['camera'], img_shifted)

print("=== Advanced Multi-Scale Structural Assessment ===")
print(f"AWGN Noisy Image:     MS-SSIM = {ms_ssim_noisy:.4f}")
print(f"1-Pixel Shift Image:  MS-SSIM = {ms_ssim_shifted:.4f}")
